<a href="https://colab.research.google.com/github/rozmar/Analyzing-Open-Neuroscience-Data-SZTE-2027/blob/main/Modul%202/01_Anatomy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Anatomy

The cortex is full of different kinds of neurons. One of the oldest ways to tell them apart is by their **shape**:
where the cell body sits, how far the dendrites reach, and where the axon goes. Since Ramón y Cajal,
neuroscientists have drawn neurons; today they are traced on a computer and saved as files.

In this course we work with **Patch-seq** data from the Allen Institute. In a Patch-seq experiment one neuron is
recorded with a glass pipette (its *electrophysiology*), filled with a dye so that its shape can be reconstructed
(its *anatomy*), and finally its content is sucked out and sequenced (its *transcriptome*). So for many cells we have
all three kinds of data. Today we look at the anatomy.

**Today we will:**
1. get to know `cells.csv`, the table that lists every cell of the course,
2. look at where in the cortex the different cell types sit,
3. open a reconstruction file of a single neuron and draw it,
4. measure how long its dendrites and axon are, and check our number against the Allen Institute's,
5. compare the shape of many neurons, in mouse and in human,
6. leave the brain slice behind and look at complete neurons that span the whole brain.

**From the bootcamp you will need:** reading tables with `pandas`, selecting rows and columns, `groupby` and
`merge`, `numpy` arithmetic, and plotting with `matplotlib`.

### How to use this notebook

**Seven** code cells contain a small mistake on purpose: 🐞 marks a bug to fix, ✏️ marks something to fill in
(replace `FILL_IN`). Each one is numbered (1/7 … 7/7), and its comment tells you which bootcamp notebook (in
`Modul 1`) and which section to look back at. The fixes are short: one word or one line.

Some of these cells end with a check, `assert ..., "message"`. While something is still wrong, the check stops with
the message; when it is right, the cell runs without complaint. Don't change the checks!


## 0. Setup

The cell below prepares everything. On Google Colab it downloads the course repository and builds the course
data (this runs `00_Build_Course_Data.ipynb` and takes about a minute). On your own computer it only builds the data
if it is not there yet.

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    REPO = "/content/Analyzing-Open-Neuroscience-Data-SZTE-2027"
    if not os.path.exists(REPO):
        !git clone --depth 1 https://github.com/rozmar/Analyzing-Open-Neuroscience-Data-SZTE-2027.git {REPO}
    os.chdir(REPO + "/Modul 2")

DATA = Path("data")
if not (DATA / "cells.csv").exists():
    print("Building the course data, this takes about a minute...")
    %run 00_Build_Course_Data.ipynb
print("Course data is in:", DATA.resolve())

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests

# one colour per subclass, used in every figure of the course
COLORS = {"Pvalb": "#D93137", "Sst": "#FF9900", "Sst Chodl": "#B1B10C", "Vip": "#B864CC",
          "Lamp5": "#DA808C", "Sncg": "#6B8E23", "Lamp5/Pax6/Other": "#DA808C"}

## 1. The cells table

`cells.csv` is the "phone book" of the course: **one row is one neuron**, from either of the two datasets:

* **mouse**: 4,435 GABAergic interneurons from the visual cortex (Gouwens, Sorensen et al. 2020, *Cell*)
* **human**: 778 GABAergic interneurons from neurosurgical tissue (Lee, Dalley et al. 2023, *Science*)

Every other table of the course (features, gene expression) has a `cell_id` column, so it can be joined to this one.

In [ ]:
cells = pd.read_csv(DATA / "cells.csv", low_memory=False)
cells.shape

In [ ]:
cells.head()

That is a lot of columns. They fall into a few groups; these are the ones we use most:

| group | columns | meaning |
|---|---|---|
| who is it? | `cell_id`, `species`, `specimen_id`, `donor_id` | identifiers |
| where was it? | `area`, `layer`, `soma_depth_normalized` | brain area, cortical layer, depth of the cell body |
| what type is it? | `class` > `subclass` > `t_type`, and `met_type` | cell type labels, from broad to fine |
| what data do we have? | `has_morphology`, `has_ephys`, `has_transcriptome` | True / False |
| where are the raw files? | `swc_url`, `nwb_url` | download links for the reconstruction and the recording |

Every column is described in `data/DATA_DICTIONARY.md`.

**About the cell type labels.** The labels come from the transcriptome: each cell was matched to a reference
"family tree" of cortical cell types. A *subclass* (e.g. `Pvalb`) contains several *t-types*
(e.g. `Pvalb Reln Itm2a`). For about 500 mouse cells, Gouwens et al. also defined *MET-types*, which combine
morphology, electrophysiology and transcriptomics. Where these labels come from exactly is the topic of lesson 11;
today we just use them.

### 1.1 How many cells, of which kind?

In [ ]:
cells["species"].value_counts()

In [ ]:
pd.crosstab(cells["subclass"], cells["species"])

The two datasets name their subclasses differently (mouse: `Pvalb`, human: `PVALB`), and the human data merges
some smaller groups. The column `subclass_harmonized` puts both species on the same four names, so that we can
compare them:

In [ ]:
pd.crosstab(cells["subclass_harmonized"], cells["species"])

### 1.2 Which data exists for each cell?

The `has_...` columns are `True`/`False`. Summing a True/False column counts the `True` values.

In [ ]:
# 🐞 FIX ME 1/7: we want to COUNT the cells, but we get fractions. Read the sentence above this cell.
# Bootcamp: 07_Introduction_To_Pandas, "other built in summary statistic functions"
counts = cells.groupby("species")[["has_morphology", "has_ephys", "has_transcriptome"]].mean()
assert counts.loc["human", "has_morphology"] == 140, "these should be numbers of cells"
counts

Morphology is the rarest: reconstructing a neuron by hand takes many hours, so only the best-filled cells were
reconstructed. Today we only need these cells:

In [ ]:
morph_cells = cells[cells["has_morphology"]]
pd.crosstab(morph_cells["subclass_harmonized"], morph_cells["species"])

### 1.3 Everything about one cell

`set_index("cell_id")` lets us look up a cell by its id with `.loc`:

In [ ]:
cells_by_id = cells.set_index("cell_id")
cells_by_id.loc["mouse_601506507"]

Many values are `NaN` ("not a number", i.e. missing): some columns only exist for one species, and not every
measurement was made for every cell.

## 2. Where do the cells sit in the cortex?

The cortex is a sheet of tissue about 1 mm thick in the mouse (2–3 mm in human). Its outer surface is the
**pia**, its inner border is the **white matter**, and in between it is organised into **layers** (L1 at the
top, L6 at the bottom).

For the mouse cells with a reconstruction we know the depth of the cell body: `soma_depth_normalized`
is 0 at the pia and 1 at the white matter. Do the subclasses prefer different depths?

In [ ]:
mouse = cells[cells["species"] == "mouse"]
subclasses = ["Lamp5", "Sncg", "Vip", "Sst", "Sst Chodl", "Pvalb"]

rng = np.random.default_rng(0)          # random numbers, to spread the dots sideways
fig, ax = plt.subplots(figsize=(7, 4))
for i, sc in enumerate(subclasses):
    depth = mouse.loc[mouse["subclass"] == sc, "soma_depth_normalized"].dropna()
    x = i + rng.uniform(-0.25, 0.25, len(depth))
    ax.scatter(x, depth, s=8, color=COLORS[sc], alpha=0.6)
    ax.plot([i - 0.3, i + 0.3], [depth.median()] * 2, color="black")   # median as a line

ax.set_xticks(range(len(subclasses)))
ax.set_xticklabels(subclasses)
ax.set_ylabel("soma depth (0 = pia, 1 = white matter)")
ax.invert_yaxis()                        # pia at the top, like in the brain
ax.set_title("Mouse: where is the cell body?")
plt.show()

**Look at the plot.** Which subclasses live close to the pia, and which ones deep in the cortex?

For the human cells we only know the layer:

In [ ]:
human = cells[cells["species"] == "human"]
pd.crosstab(human["layer"], human["subclass"])

## 3. One neuron, from a file

After the recording, the neuron was filled with **biocytin**, stained, and imaged under a microscope. Then someone
traced it by hand: they clicked along every branch, and the computer stored the clicks as a **tree of points**.
Such a tree is saved as an **SWC file**. It is a plain text file, one point per line:

```
# id  type  x       y        z      radius  parent
1     1     0.0     0.0      0.0    5.2     -1
2     3     2.1     -1.5     0.3    0.6     1
3     3     4.0     -3.2     0.5    0.5     2
```

| column | meaning |
|---|---|
| `id` | number of the point |
| `type` | 1 = soma, 2 = axon, 3 = (basal) dendrite, 4 = apical dendrite |
| `x`, `y`, `z` | position in µm |
| `radius` | thickness of the branch at that point, in µm |
| `parent` | `id` of the point it connects to (`-1`: no parent, this is the root, the soma) |

So the whole shape is stored with just one rule: *every point is connected to its parent*.

We use the "transformed" files, in which the neuron is rotated so that the **y axis is perpendicular to the
cortical surface**.

The files live on the [Brain Image Library](https://www.brainimagelibrary.org). **If it is down**, set
`USE_BACKUP_SWC = True` below: then we use three of our cells (one Pvalb, one Sst, one Vip) whose reconstructions
the Allen Institute also publishes on GitHub, as example data of its
[skeleton_keys](https://github.com/AllenInstitute/skeleton_keys) software.

In [ ]:
USE_BACKUP_SWC = True

BACKUP_SWC_URL = ("https://raw.githubusercontent.com/AllenInstitute/skeleton_keys/"
                  "9da02cc8e3b437fddb0d367d8dd6acef5d42bf4e/example_data/layer_aligned_swcs/{}.swc")

if USE_BACKUP_SWC:
    cells["swc_url"] = None
    for specimen_id in [606271263, 740135032, 694146546]:        # Pvalb, Sst, Vip
        cells.loc[cells["specimen_id"] == specimen_id, "swc_url"] = BACKUP_SWC_URL.format(specimen_id)
    cells_by_id = cells.set_index("cell_id")
    morph_cells = cells[cells["has_morphology"]]

Let's pick one reconstructed mouse Pvalb cell:

In [ ]:
has_file = morph_cells[morph_cells["swc_url"].notna()]
if len(has_file) == 0:
    raise RuntimeError("No SWC links in cells.csv. Rebuild the course data (00_Build_Course_Data.ipynb) "
                       "and check its validation report for the 'mouse manifest' line.")

example = has_file[has_file["subclass"] == "Pvalb"].iloc[0]
example_id = example["cell_id"]
example[["cell_id", "subclass", "t_type", "met_type", "layer", "swc_url"]]

This small function downloads the file
once and keeps a copy in `data/swc/`, so the second time it is instant:

In [ ]:
SWC_DIR = DATA / "swc"
SWC_DIR.mkdir(exist_ok=True)

def download_swc(url):
    """Download an SWC file (once) and return the path of the local copy."""
    path = SWC_DIR / url.split("/")[-1]
    if not path.exists():
        response = requests.get(url, timeout=60)
        response.raise_for_status()           # stop with an error if the download failed
        path.write_bytes(response.content)
    return path

swc_path = download_swc(example["swc_url"])
swc_path

It is just text. Let's print the first 12 lines:

In [ ]:
with open(swc_path) as f:
    for i, line in enumerate(f):
        print(line.rstrip())
        if i == 11:
            break

Lines starting with `#` are comments. The rest is a table with spaces between the columns, so `pd.read_csv`
can read it if we tell it three things: the columns are separated by spaces (`sep=r"\s+"`), lines starting
with `#` should be skipped (`comment="#"`), and what the column names are (`names=...`).

In [ ]:
SWC_COLUMNS = ["id", "type", "x", "y", "z", "radius", "parent"]

swc = pd.read_csv(swc_path, sep=r"\s+", comment="#", names=SWC_COLUMNS)
print(swc.shape)
swc.head()

The `type` numbers are easier to read as names. `.map()` replaces every value using a dictionary:

In [ ]:
# ✏️ FILL IN 2/7: replace FILL_IN with the right names (strings), using the table at the start of section 3.
# Bootcamp: 01_Basic_Python_I_Object_and_Data_Structures, the part on dictionaries (Exercise 1.8)
TYPE_NAMES = {1: "soma", 2: FILL_IN, 3: FILL_IN, 4: "apical dendrite"}
swc["type_name"] = swc["type"].map(TYPE_NAMES)
swc["type_name"].value_counts()

The root of the tree is the point without a parent:

In [ ]:
soma = swc[swc["parent"] == -1]
soma

## 4. Drawing the neuron

### 4.1 Just the points

The simplest picture: every point as a dot, coloured by type. We move the soma to (0, 0) by subtracting its
position, and use `set_aspect("equal")` so that 100 µm is equally long in x and in y.

In these files the y axis already points towards the pia, so `PIA_UP = True` (no flipping needed).

In [ ]:
PIA_UP = True          # True: y in the file already points towards the pia

def centre_on_soma(swc):
    """Shift the coordinates so that the soma is at (0, 0, 0), with the pia upwards."""
    soma = swc[swc["parent"] == -1].iloc[0]
    swc = swc.copy()
    for col in ["x", "y", "z"]:
        swc[col] = swc[col] - soma[col]
    if not PIA_UP:
        swc["y"] = -swc["y"]
    return swc

swc = centre_on_soma(swc)

fig, ax = plt.subplots(figsize=(5, 6))
for name, color in [("dendrite", "black"), ("axon", "tab:red")]:
    part = swc[swc["type_name"] == name]
    ax.scatter(part["x"], part["y"], s=0.5, color=color, label=name)
ax.scatter(0, 0, s=60, color="tab:blue", label="soma")
ax.set_aspect("equal")
ax.set_xlabel("x (µm)")
ax.set_ylabel("y (µm), pia is up")
ax.legend()
plt.show()

### 4.2 Connecting the dots

To draw real branches, every point must be connected to its parent with a line. The trick: make a table in
which each row holds a point **and** the position of its parent. That is a `merge`: we look up each point's
`parent` in a copy of the table where the columns are renamed to `parent`, `px`, `py`, `pz`.

In [ ]:
def make_segments(swc):
    """One row per branch piece: a point (x, y, z) and its parent (px, py, pz)."""
    parents = swc[["id", "x", "y", "z"]].rename(
        columns={"id": "parent", "x": "px", "y": "py", "z": "pz"})
    # 🐞 FIX ME 3/7: this function makes the table, but gives back nothing (None). What is missing?
    # Bootcamp: 02_Basic_Python_II_Control_Flow_and_Functions, "Functions"
    swc.merge(parents, on="parent")     # the soma has no parent, so it drops out

segments = make_segments(swc)
segments[["id", "type_name", "x", "y", "parent", "px", "py"]].head()

Now each row is one short line, from `(px, py)` to `(x, y)`. We could call `plt.plot` once per row, but that is
slow for thousands of rows. Instead we use a matplotlib trick: in one long list of points, `NaN` means
"lift the pen". So for two pieces we draw `[px1, x1, NaN, px2, x2, NaN]`:

In [ ]:
def draw(segments, ax, color, xcol="x", ycol="y"):
    """Draw all branch pieces with a single plot call (NaN = lift the pen)."""
    nan = np.full(len(segments), np.nan)
    xs = np.column_stack([segments["p" + xcol], segments[xcol], nan]).ravel()
    ys = np.column_stack([segments["p" + ycol], segments[ycol], nan]).ravel()
    ax.plot(xs, ys, color=color, linewidth=0.6)

We draw the neuron from the front (x–y) and from the side (z–y):

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 6), sharey=True)
for ax, xcol in zip(axes, ["x", "z"]):
    draw(segments[segments["type_name"] == "axon"], ax, "tab:red", xcol=xcol)
    draw(segments[segments["type_name"] == "dendrite"], ax, "black", xcol=xcol)
    ax.scatter(0, 0, s=40, color="tab:blue", zorder=3)
    ax.set_aspect("equal")
    ax.set_xlabel(f"{xcol} (µm)")
axes[0].set_ylabel("y (µm), pia is up")
axes[0].set_title("front view")
axes[1].set_title("side view")
fig.suptitle(f"{example_id}: {example['t_type']}  (axon red, dendrite black)")
plt.show()

### 4.3 In 3D

The file also has a `z` coordinate, so we can look at the neuron from every side. **plotly** makes figures that
you can rotate (drag with the mouse) and zoom (scroll). The NaN trick from `draw()` works here too, so
`trace3d()` is the same idea in 3D. Up in the figure is `y`, the direction of the pia.

In [ ]:
import plotly.graph_objects as go
from matplotlib.colors import to_hex                  # plotly does not know matplotlib's "tab:" colours

def trace3d(segments, color, name, x="x", y="z", z="y"):
    """All branch pieces as one plotly 3D line (NaN = lift the pen). By default the file's y is drawn upwards."""
    nan = np.full(len(segments), np.nan)
    line = lambda c: np.column_stack([segments["p" + c], segments[c], nan]).ravel()
    return go.Scatter3d(x=line(x), y=line(y), z=line(z), mode="lines", name=name,
                        line=dict(color=to_hex(color), width=2))

fig = go.Figure([trace3d(segments[segments["type_name"] == "axon"], "tab:red", "axon"),
                 trace3d(segments[segments["type_name"] == "dendrite"], "black", "dendrite"),
                 go.Scatter3d(x=[0], y=[0], z=[0], mode="markers", name="soma",
                              marker=dict(size=6, color=to_hex("tab:blue")))])
fig.update_layout(scene=dict(aspectmode="data", xaxis_title="x (µm)", yaxis_title="z (µm)",
                             zaxis_title="y (µm), pia is up"),
                  title=f"{example_id}: drag to rotate, scroll to zoom", height=600)
fig.show()

**Look at the picture.**
* Which part is bigger, the axon or the dendrites?
* Does the axon stay close to the soma, or does it travel to other layers? Pvalb cells are typically
  **basket cells**: their axon wraps around the cell bodies of nearby neurons.
* The side view is thinner than the front view. Why? (Hint: the brain slice was about 300 µm thick.)

## 5. How long are the dendrites?

Every branch piece is a straight line between two points, so its length comes from Pythagoras in 3D:

$$\text{length} = \sqrt{(x - p_x)^2 + (y - p_y)^2 + (z - p_z)^2}$$

With numpy we compute this for all pieces at once, without a loop:

In [ ]:
# 🐞 FIX ME 4/7: one of the three terms is not squared. Compare the code with the equation above.
# Bootcamp: 04_Introduction_To_Numpy, "Arithmetic with arrays" (the power operator)
segments["length"] = np.sqrt((segments["x"] - segments["px"]) ** 2
                             + (segments["y"] - segments["py"]) * 2
                             + (segments["z"] - segments["pz"]) ** 2)
assert segments["length"].notna().all(), "some lengths are NaN (square root of a negative number?)"
segments["length"].describe()

The total length of each part is the sum of its pieces. The coordinates are in µm; we divide by 1000 to get mm:

In [ ]:
total_mm = segments.groupby("type_name")["length"].sum() / 1000
total_mm

### 5.1 Check against the Allen Institute

The Allen Institute has already computed about 40 morphology features for every reconstructed cell. They are in
`mouse_morph_features.csv`. Is our number the same as theirs?

In [ ]:
mouse_morph = pd.read_csv(DATA / "mouse_morph_features.csv").set_index("cell_id")

print("dendrite  ours: %.2f mm   Allen: %.2f mm" % (
    total_mm["dendrite"], mouse_morph.loc[example_id, "basal_dendrite_total_length_mm"]))
print("axon      ours: %.2f mm   Allen: %.2f mm" % (
    total_mm["axon"], mouse_morph.loc[example_id, "axon_total_length_mm"]))

Checking your own result against an independent calculation is one of the most useful habits in data analysis.
If the numbers agree, both are probably right. Small differences can come from details: for example, brain
tissue shrinks a little when it is fixed and stained, and some versions of the files are corrected for this.

*(Interneurons have no apical dendrite, so all their dendrites are "basal dendrites" in the Allen table.)*

## 6. Any neuron, with functions

We now put the steps of sections 3–5 into functions, so that we can repeat them for any cell with one line.

In [ ]:
def load_neuron(cell_id):
    """Download, read and centre the SWC file of a cell; return its branch pieces with their lengths."""
    url = cells_by_id.loc[cell_id, "swc_url"]
    swc = pd.read_csv(download_swc(url), sep=r"\s+", comment="#", names=SWC_COLUMNS)
    swc["type_name"] = swc["type"].map(TYPE_NAMES)
    segments = make_segments(centre_on_soma(swc))
    segments["length"] = np.sqrt((segments["x"] - segments["px"]) ** 2
                                 + (segments["y"] - segments["py"]) ** 2
                                 + (segments["z"] - segments["pz"]) ** 2)
    return segments


def total_length_mm(segments, type_name):
    """Total length of one part of the neuron ('dendrite' or 'axon'), in mm."""
    return segments.loc[segments["type_name"] == type_name, "length"].sum() / 1000

Let's take two cells from each of three subclasses:

In [ ]:
chosen = []
for sc in ["Pvalb", "Sst", "Vip"]:
    ids = has_file.loc[has_file["subclass"] == sc, "cell_id"]
    # 🐞 FIX ME 5/7: we want 2 cells from EACH subclass, but in the end the list only has the last 2.
    # Bootcamp: 02_Basic_Python_II_Control_Flow_and_Functions, "List Comprehensions" (the long way, with a loop)
    chosen = list(ids.iloc[:2])
assert len({cells_by_id.loc[c, "subclass"] for c in chosen}) == 3, "we want cells from all three subclasses"
chosen

In [ ]:
neurons = {}          # cell_id -> branch pieces
rows = []             # one dictionary per cell, for a results table
for cell_id in chosen:
    seg = load_neuron(cell_id)
    neurons[cell_id] = seg
    rows.append({"cell_id": cell_id,
                 "subclass": cells_by_id.loc[cell_id, "subclass"],
                 "dendrite_ours_mm": total_length_mm(seg, "dendrite"),
                 "dendrite_allen_mm": mouse_morph.loc[cell_id, "basal_dendrite_total_length_mm"]})

results = pd.DataFrame(rows)
results

If ours and Allen's agree, the points lie on the diagonal:

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4))
ax.scatter(results["dendrite_allen_mm"], results["dendrite_ours_mm"],
           color=[COLORS[sc] for sc in results["subclass"]])
lim = [0, results[["dendrite_allen_mm", "dendrite_ours_mm"]].max().max() * 1.1]
ax.plot(lim, lim, "--", color="gray")              # the diagonal: ours = Allen
ax.set_xlabel("Allen (mm)")
ax.set_ylabel("ours (mm)")
ax.set_title("Total dendrite length")
plt.show()

And a small gallery, all neurons on the same scale:

In [ ]:
fig, axes = plt.subplots(1, len(chosen), figsize=(2.5 * len(chosen), 5), sharex=True, sharey=True)
for ax, cell_id in zip(axes, chosen):
    seg = neurons[cell_id]
    sc = cells_by_id.loc[cell_id, "subclass"]
    draw(seg[seg["type_name"] == "axon"], ax, COLORS[sc])
    draw(seg[seg["type_name"] == "dendrite"], ax, "black")
    ax.set_aspect("equal")
    ax.set_title(sc)
    ax.axis("off")
plt.show()

**Look at the gallery.** How do the axons of Pvalb, Sst and Vip cells differ? Sst cells are often
**Martinotti cells**, whose axon climbs up to layer 1; Vip cells are often narrow and vertical.

## 7. Many neurons: the precomputed features

Downloading and measuring hundreds of files would take a while, so for the whole dataset we use the
features that the Allen Institute has already computed. Each row of `mouse_morph_features.csv` is one cell;
we add the subclass from `cells.csv` with a `merge` on `cell_id`.

In [ ]:
mouse_morph = pd.read_csv(DATA / "mouse_morph_features.csv")
# ✏️ FILL IN 6/7: which column do both tables have? (The text above tells you.)
# Bootcamp: 07_Introduction_To_Pandas, "Merging DataFrames"
mouse_morph = mouse_morph.merge(cells[["cell_id", "subclass", "subclass_harmonized"]], on=FILL_IN)
print(mouse_morph.shape)
mouse_morph.columns.tolist()

A plot we will use often: one feature, one column of dots per subclass. Let's make it a function.

In [ ]:
def strip_plot(df, feature, group, order, ax, offset=0.0, marker="o"):
    """One column of dots per group, with the median as a black line."""
    rng = np.random.default_rng(0)
    for i, g in enumerate(order):
        values = df.loc[df[group] == g, feature].dropna()
        x = i + offset + rng.uniform(-0.15, 0.15, len(values))
        ax.scatter(x, values, s=10, color=COLORS[g], alpha=0.6, marker=marker)
        ax.plot([i + offset - 0.2, i + offset + 0.2], [values.median()] * 2, color="black")
    ax.set_xticks(range(len(order)))
    ax.set_xticklabels(order)
    ax.set_ylabel(feature)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
strip_plot(mouse_morph, "basal_dendrite_total_length_mm", "subclass", subclasses, axes[0])
strip_plot(mouse_morph, "axon_total_length_mm", "subclass", subclasses, axes[1])
axes[0].set_title("Mouse: dendrite length")
axes[1].set_title("Mouse: axon length")
plt.show()

**Questions.** Which subclass has the longest axon? The shortest? Are the dendrites as different between
subclasses as the axons are?

Shape is more than length. Let's look at how wide (`extent_x`) and how tall (`extent_y`) the dendrites are:

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
for sc in subclasses:
    part = mouse_morph[mouse_morph["subclass"] == sc]
    ax.scatter(part["basal_dendrite_extent_x_mm"], part["basal_dendrite_extent_y_mm"],
               s=10, color=COLORS[sc], label=sc, alpha=0.7)
ax.plot([0, 0.8], [0, 0.8], "--", color="gray")      # on this line: as wide as tall
ax.set_xlabel("dendrite width (mm)")
ax.set_ylabel("dendrite height (mm)")
ax.legend()
plt.show()

Which subclass has tall, narrow dendrites (above the diagonal)? Does that match what you saw in the gallery?

### 7.1 Mouse and human

The human table has the same kind of features, but **careful: the human lengths are in µm, the mouse lengths
in mm.** (Always check the units before comparing!) We convert the human values to mm:

In [ ]:
human_morph = pd.read_csv(DATA / "human_morph_features.csv")
human_morph = human_morph.merge(cells[["cell_id", "subclass_harmonized"]], on="cell_id")
human_morph["basal_dendrite_total_length_mm"] = human_morph["basal_dendrite_total_length"] / 1000
human_morph["axon_total_length_mm"] = human_morph["axon_total_length"] / 1000
print(human_morph.shape)

In [ ]:
harmonized = ["Lamp5/Pax6/Other", "Vip", "Sst", "Pvalb"]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, feature in zip(axes, ["basal_dendrite_total_length_mm", "axon_total_length_mm"]):
    strip_plot(mouse_morph, feature, "subclass_harmonized", harmonized, ax, offset=-0.2, marker="o")
    strip_plot(human_morph, feature, "subclass_harmonized", harmonized, ax, offset=+0.2, marker="^")
    ax.set_title(feature + "\n(left: mouse ●, right: human ▲)")
plt.show()

**Questions to think about.**
* Are human interneurons bigger than mouse interneurons? For every subclass?
* The human cortex is 2–3 times thicker than the mouse cortex. Would you expect bigger neurons?
* Before drawing conclusions, what else differs between the two datasets? (Hint: look at the `preparation` and
  `disease` columns of the human cells in `cells.csv`.)

## 8. Beyond the slice: whole neurons

All the cells we looked at today are **local interneurons**: their axon stays within about half a millimetre of
the cell body. But many neurons are **projection neurons**, whose axon leaves its home area and travels to
other parts of the brain. In a 300 µm thick slice such an axon is simply cut off.

To see a projection neuron completely, you need the whole brain. The **MouseLight** project at Janelia labelled a
few neurons per mouse brain, imaged the entire brain under a microscope, and traced each neuron from start to end
(Winnubst et al. 2019, *Cell*). The reconstructions are placed in the **Allen Common Coordinate Framework (CCF)**,
a shared 3D map of the mouse brain in µm, so neurons from different mice can be shown in one brain.

We will download two of these neurons, AA0184 and AA0261. Both have their cell body in **layer 5 of the primary
motor cortex** (MOp5), yet they turn out to be very different.

All MouseLight neurons can be browsed and downloaded in the
[MouseLight Neuron Browser](https://ml-neuronbrowser.janelia.org). Its "download" button sends a request to the
address `https://ml-neuronbrowser.janelia.org/export`, and we can send the same request from Python. This time
it is a **POST** request: instead of just asking for a file, we send some data along (which neurons, in which
format). The answer comes as **JSON**, a text format that Python reads straight into dictionaries and lists.

If the MouseLight server does not answer, the function falls back to a copy of the same neurons that the
[Open Source Brain](https://github.com/OpenSourceBrain/MouseLightShowcase) project keeps on GitHub.

*MouseLight data: CC BY-NC 4.0, Janelia Research Campus. Winnubst et al. (2019) Cell 179:268.*

In [ ]:
import json

MOUSELIGHT_EXPORT = "https://ml-neuronbrowser.janelia.org/export"
# backup: the same neurons exported from MouseLight in 2017/18, on GitHub (Open Source Brain)
BACKUP_MOUSELIGHT = ("https://raw.githubusercontent.com/OpenSourceBrain/MouseLightShowcase/"
                     "ec596c2164862f7abef1573a3934b74c074df141/JSON/{}")
BACKUP_FILES = {"AA0184": "MOp2.json", "AA0261": "MOp.json"}

WHOLE_DIR = DATA / "whole_neurons"
WHOLE_DIR.mkdir(exist_ok=True)

def download_mouselight(neuron_id):
    """Ask the MouseLight server for one neuron; if it does not answer, use the GitHub backup."""
    try:
        # format 1 = JSON, ccfVersion 1 = Allen CCF version 3 (the same atlas we draw below)
        request = {"ids": [neuron_id], "format": 1, "ccfVersion": 1}
        response = requests.post(MOUSELIGHT_EXPORT, json=request, timeout=60)
        response.raise_for_status()
        return response.json()["contents"]["neurons"][0]
    except Exception as error:
        print(f"MouseLight server did not answer ({type(error).__name__}), using the GitHub backup")
        response = requests.get(BACKUP_MOUSELIGHT.format(BACKUP_FILES[neuron_id]), timeout=120)
        response.raise_for_status()
        for neuron in response.json()["neurons"]:      # the backup files hold several neurons
            if neuron["idString"] == neuron_id:
                # these older files use another axis order (x = left-right, z = front-back): swap x and z
                for point in [neuron["soma"]] + neuron["axon"] + neuron["dendrite"]:
                    point["x"], point["z"] = point["z"], point["x"]
                return neuron


def load_mouselight(neuron_id):
    """Download a MouseLight neuron once, keep a copy in data/whole_neurons/, return it as a dictionary."""
    path = WHOLE_DIR / f"{neuron_id}.json"
    if not path.exists():
        path.write_text(json.dumps(download_mouselight(neuron_id)))
    with open(path) as f:
        return json.load(f)

neuron = load_mouselight("AA0184")
print(neuron.keys())
# 🐞 FIX ME 7/7: KeyError! Look at the keys printed by the line above.
# Bootcamp: 01_Basic_Python_I_Object_and_Data_Structures, the part on dictionaries (accessing a missing key)
print(neuron["Soma"])
print(neuron["axon"][:2])

`neuron["axon"]` is a list of points, and every point is a dictionary with the same information as a row of an
SWC file, just with different names (`sampleNumber` = `id`, `structureIdentifier` = `type`,
`parentNumber` = `parent`). A list of dictionaries becomes a table with `pd.DataFrame`. After renaming the
columns, our `make_segments` function from section 4 works without any change.

There is one bonus: every point also has an `allenId`, the brain region it lies in. The list
`neuron["allenInformation"]` translates these ids into region names.

The coordinates are in µm and follow the axis order of the Allen atlas: `x` goes from front (anterior) to back
(posterior), `y` from top (dorsal) to bottom (ventral), and `z` from left to right.

In [ ]:
MOUSELIGHT_NAMES = {"sampleNumber": "id", "structureIdentifier": "type", "parentNumber": "parent"}

def mouselight_segments(neuron):
    """Branch pieces of the axon and dendrite of a MouseLight neuron, with length and brain region."""
    region_names = {r["allenId"]: r["acronym"] for r in neuron["allenInformation"]}
    parts = []
    for part in ["axon", "dendrite"]:
        points = pd.DataFrame(neuron[part]).rename(columns=MOUSELIGHT_NAMES)
        segments = make_segments(points)
        segments["type_name"] = part
        parts.append(segments)
    segments = pd.concat(parts, ignore_index=True)
    segments["length"] = np.sqrt((segments["x"] - segments["px"]) ** 2
                                 + (segments["y"] - segments["py"]) ** 2
                                 + (segments["z"] - segments["pz"]) ** 2)
    segments["region"] = segments["allenId"].map(region_names)
    return segments

whole_ids = ["AA0184", "AA0261"]
whole = {nid: mouselight_segments(load_mouselight(nid)) for nid in whole_ids}

for nid, seg in whole.items():
    print(f"{nid}: axon {total_length_mm(seg, 'axon'):.0f} mm, dendrite {total_length_mm(seg, 'dendrite'):.1f} mm")

Compare this with the interneurons of section 7:

In [ ]:
lengths = {"mouse interneurons\n(median)": mouse_morph["axon_total_length_mm"].median()}
for nid, seg in whole.items():
    lengths[nid] = total_length_mm(seg, "axon")

fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(list(lengths.keys()), list(lengths.values()), color=["gray", "tab:blue", "tab:green"])
ax.set_ylabel("total axon length (mm)")
plt.show()

### 8.1 The brain around them

To see where these axons go, we draw them inside the outline of the mouse brain. The Allen mouse brain atlas is
available through the **BrainGlobe** Python package; the first time, it downloads the atlas.

In [ ]:
try:
    from brainglobe_atlasapi import BrainGlobeAtlas
except ImportError:
    %pip install -q brainglobe-atlasapi
    from brainglobe_atlasapi import BrainGlobeAtlas

atlas = BrainGlobeAtlas("allen_mouse_100um")      # Allen CCF, one voxel = 100 µm
print(atlas.annotation.shape)                     # (anterior-posterior, dorsal-ventral, left-right)

`atlas.annotation` is a 3D numpy array: every 100 µm cube of the brain contains the id of the brain region it
belongs to, and 0 outside the brain. `annotation > 0` is True inside the brain. With `.any(axis=...)` we look
"through" the brain along one axis, which gives its outline seen from above and from the side.

In [ ]:
inside = atlas.annotation > 0
top_view = inside.any(axis=1)          # seen from above: anterior-posterior x left-right
side_view = inside.any(axis=2)         # seen from the side: anterior-posterior x dorsal-ventral
ap_um, dv_um, lr_um = atlas.shape_um   # size of the atlas in µm

colors = {"AA0184": "tab:blue", "AA0261": "tab:green"}

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(top_view, extent=[0, lr_um, ap_um, 0], cmap="Greys", alpha=0.25)
axes[1].imshow(side_view.T, extent=[0, ap_um, dv_um, 0], cmap="Greys", alpha=0.25)

for nid, seg in whole.items():
    axon = seg[seg["type_name"] == "axon"]
    # MouseLight uses the atlas axis order (CCFv3): x = front-back, y = top-bottom, z = left-right
    draw(axon, axes[0], colors[nid], xcol="z", ycol="x")     # from above: left-right across, front-back down
    draw(axon, axes[1], colors[nid], xcol="x", ycol="y")     # from the side: front-back across, top-bottom down
axes[0].set_title("from above (front is up)")
axes[1].set_title("from the side (front is left)")
for ax in axes:
    ax.set_xlabel("µm")
fig.suptitle("AA0184 (blue) and AA0261 (green): two layer 5 motor cortex neurons")
plt.show()

**Look at the picture.** Which neuron sends its axon to the other half of the brain? Which one goes deep, towards
the back and bottom of the brain? For scale: the axon of the Pvalb cell from section 4 would fit into a few
pixels of this picture.

In [ ]:
root = atlas.mesh_from_structure("root")              # brain surface, µm: (front-back, top-bottom, left-right)
ap, dv, lr = root.points.T
i, j, k = root.cells_dict["triangle"].T

fig = go.Figure(go.Mesh3d(x=ap, y=lr, z=dv, i=i, j=j, k=k, name="brain",
                          color="lightgray", opacity=0.1, hoverinfo="skip"))
for nid, seg in whole.items():
    # MouseLight uses the atlas axis order (CCFv3): x = front-back, y = top-bottom, z = left-right
    fig.add_trace(trace3d(seg[seg["type_name"] == "axon"], colors[nid], nid))
fig.update_layout(scene=dict(aspectmode="data", xaxis_title="front-back (µm)", yaxis_title="left-right (µm)",
                             zaxis=dict(title="top-bottom (µm)", autorange="reversed")),  # atlas y grows downwards
                  height=600)
fig.show()

### 8.2 Where does the axon go?

Because every piece of axon knows its brain region, the question "how much axon is in each region?" is a single
`groupby`:

In [ ]:
region_full_names = {}
for nid in whole_ids:
    for r in load_mouselight(nid)["allenInformation"]:
        region_full_names[r["acronym"]] = r["name"]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, nid in zip(axes, whole_ids):
    axon = whole[nid][whole[nid]["type_name"] == "axon"]
    per_region = axon.groupby("region")["length"].sum().sort_values() / 1000
    top10 = per_region.tail(10)
    labels = [f"{acr}: {region_full_names.get(acr, '')[:35]}" for acr in top10.index]
    ax.barh(labels, top10.values, color=colors[nid])
    ax.set_xlabel("axon length (mm)")
    ax.set_title(nid)
plt.tight_layout()
plt.show()

These two neurons are the two classic kinds of layer 5 projection neurons:

* **IT** (intratelencephalic) neurons, like AA0184, stay in the forebrain: their axon goes to other cortical areas,
  to the striatum (`CP`, caudoputamen), and across the corpus callosum (`cc`) to the other hemisphere.
* **PT** (pyramidal tract) neurons, like AA0261, send their axon down to the thalamus, midbrain and brainstem
  (e.g. `MRN`, midbrain reticular nucleus); many reach the spinal cord.

Same area, same layer, completely different targets: where the axon goes is a big part of what a cell type is.

*Note: if the GitHub backup was used, the neurons are in an older version of the Allen coordinate framework
than the atlas outline, so small mismatches at the edges are expected.* Many more neurons can be browsed at
[ml-neuronbrowser.janelia.org](https://ml-neuronbrowser.janelia.org).

## Summary

* `cells.csv` lists every cell; all other tables join to it on `cell_id`.
* A reconstruction is a tree of points saved as an SWC file: a plain text table in which every point knows its parent.
* From the tree we drew the neuron and measured its length, and checked our result against the Allen Institute's.
* Different subclasses have different shapes, especially their axons.
* Interneurons are local; projection neurons, reconstructed in whole brains, send their axon across the brain.

**Next lesson: electrophysiology.** Same cells, but this time we look at how they fire.